<b>Group Number:</b>
<br><b>Name Group Member 1:</b>
<br><b>u-Kürzel Group Member 1:</b>
<br><b>Name Group Member 2:</b>
<br><b>u-Kürzel Group Member 2:</b>

# Unit 2: Data Cleaning & Feature Engineering

## Introduction


In Unit 1, we loaded the Bank Marketing dataset and performed Exploratory Data Analysis (EDA). We gained an initial understanding of the features, their distributions, correlations, and identified potential issues like class imbalance and the problematic 'duration' feature.

Now, in Unit 2, we transition to the crucial steps of **Data Cleaning** and **Feature Engineering/Transformation**. Raw datasets collected in the real world are rarely pristine; they often contain inconsistencies (like non-standard missing value codes), require features to be converted into formats suitable for algorithms, or can be improved by creating new, more informative features.

*   **Data Cleaning:** Focuses on identifying and addressing data quality issues. This might involve handling missing values, correcting errors, or dealing with inconsistencies.
*   **Feature Engineering:** The art and science of creating new features from existing ones, often leveraging domain knowledge or insights from EDA, to help machine learning models capture underlying patterns more effectively.
*   **Feature Transformation:** Modifying existing features to make them suitable for modelling. Key examples include encoding categorical variables (which models typically can't handle directly) and scaling numerical variables (to bring them to a common range, which benefits many algorithms).

In this unit, using the Bank Marketing dataset, we will specifically:
1.  Address the 'unknown' values present in several categorical columns.
2.  Remove the 'duration' feature identified as problematic.
3.  Briefly discuss potential outliers in numerical features.
4.  Apply One-Hot Encoding to convert categorical features into a numerical format suitable for modelling.
5.  Apply Standardization (Z-score scaling) to numerical features.
6.  Assemble the final, processed feature matrix (`X`) and target vector (`y`), saving them for subsequent units.

In [142]:
import pandas as pd  # https://pandas.pydata.org/docs/
import numpy as np  # https://numpy.org/doc/2.1/
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
)  # https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html
import os
from IPython.display import display
from typing import List

filename: str = "data/bank.csv"
bank_df: pd.DataFrame = None
bank_df = pd.read_csv(filename, sep=";")
print(
    f"Successfully loaded dataset from {filename} with shape: {bank_df.shape}")

# Create a copy to work on for cleaning and preprocessing
bank_cleaned_df = bank_df.copy()
print("Created a copy of the DataFrame for cleaning.")

Successfully loaded dataset from data/bank.csv with shape: (4521, 17)
Created a copy of the DataFrame for cleaning.


## 2.1 Handling 'Unknown' Values

From our EDA in Unit 1, we observed that several categorical columns (like 'job' and 'education') contain the string <i>unknown</i>. This likely represents missing or unrecorded information. Ignoring these can lead to errors or suboptimal model performance. We need a strategy to handle them. Common approaches include:

1.  **Imputation:** Replace the missing indicator (like <i>unknown</i> or standard `NaN` values) with a plausible value. The choice depends on the data type and distribution:
    *   **Mean Imputation:** Replace missing numerical values with the mean of the column. Suitable for roughly symmetric distributions without significant outliers. Simple, but reduces variance and distorts correlations.
    *   **Median Imputation:** Replace missing numerical values with the median of the column. More robust to outliers and skewed distributions than the mean. Still reduces variance.
    *   **Mode Imputation:** Replace missing categorical values (or sometimes numerical, if discrete) with the mode (most frequent value) of the column. This is what we'll use for our categorical <i>unknown</i>s. Simple, but can distort distributions if missingness is high.
    *   **Constant Value Imputation:** Replace missing values with a fixed constant (e.g., 0, -1, 999, or a string like "Missing"). This can be useful if the fact that the value is missing is informative in itself. It avoids making assumptions like mean/median/mode imputation but treats all missing values identically.
    *   **(Advanced) Stochastic Imputation:** For numerical data, instead of just using the mean/median, draw a random value from the observed distribution or add random noise to the mean/median imputation. Aims to preserve variance better than simple mean/median imputation.
    *   **(Advanced) Model-Based Imputation:** Use other features to predict the missing value using a machine learning model (see Deep Dive).

2.  **Treat as Separate Category:** Keep <i>unknown</i> (or create a "Missing" category from `NaN`) as its own distinct category for categorical features. This retains the information that the value was missing, which might itself be predictive. The downside is increasing dimensionality after encoding.

3.  **Remove Rows/Columns:** If missing values are present in only a tiny fraction of rows (and randomly distributed), removing those rows might be acceptable, but leads to data loss. If a column has a very high percentage of missing values and seems uninformative, removing the entire column might be considered. Generally discouraged unless the proportion is negligible or extremely high.

**Our Strategy:** For this exercise, we will use **Strategy 1: Impute with the mode** for the categorical columns containing <i>unknown</i>. It's a straightforward and common baseline approach, suitable for demonstrating the process.

<div class="alert alert-block alert-info">
    <b>Further Reading on Missing Data:</b>
    Understanding <b>why</b> data is missing can influence the best handling strategy. Concepts like MCAR (Missing Completely At Random), MAR (Missing At Random), and MNAR (Missing Not At Random) are important in more advanced scenarios. You can read more here:
    <ul>
        <li><a href="https://stefvanbuuren.name/fimd/missing-data-pattern.html" target="_blank">Flexible Imputation of Missing Data: Missing Data Pattern</a></li>
        <li><a href="https://medium.com/data-science/how-to-handle-missing-data-8646b18db0d4" target="_blank">How to Handle Missing Data (Medium)</a></li>
        <li><a href="https://scikit-learn.org/stable/modules/impute.html" target="_blank">Scikit-learn Documentation: Imputation of missing values</a></li>
    </ul>
</div>

<div class="alert alert-block alert-success">
<b>Task 2.1: Impute <i>Unknown</i> Values with Mode (2 pts)</b>

<ul>
    <li> First, identify which object-type columns in <i>bank_cleaned_df</i> contain the value <i>unknown</i>. Print the names of these columns.</li>
    <li> For each found column, calculate its mode.
        <ul><li><i>Hint:</i> Use the <code>.mode()</code> method, which returns a Series; you typically want the first element `[0]`. Be mindful of the edge case where <i>unknown</i> itself might be the most frequent value initially. If so, consider using the <i>second</i> most frequent value as the imputation target.</li></ul>
    </li>
    <li> Replace all occurrences of <i>unknown</i> in that column with the calculated mode value using the <code>.replace()</code> method.</li>
    <li> After the loop, verify that the <i>unknown</i> values have been removed by checking the counts again for the affected columns. Use an <i>assert</i> statement to confirm the count is 0 for each.</li>
</ul>
</div>

In [143]:
print("--- Handling 'Unknown' Values (Imputing with Mode) ---")

cols_with_unknown: List[str] = []
# Identify columns with 'unknown' values, and print their names
### STUDENT CODE HERE (0.75 pts)
object_cols = bank_cleaned_df.select_dtypes(include='object')

for col in object_cols:
    if object_cols[col].isin(["unknown"]).any():
        cols_with_unknown.append(col)

print(cols_with_unknown)
### STUDENT CODE until HERE

# Impute 'unknown' with the mode for the identified columns
print("\nImputing 'unknown' with mode:")
### STUDENT CODE HERE (2 pts)
for col in cols_with_unknown:
    series_mode = bank_cleaned_df[col].value_counts()


    if series_mode.iloc[0] == 'unknown'and len(series_mode) > 1 :
        value_mode = series_mode.iloc[1]
    else:
        value_mode = series_mode.iloc[0]
    
    bank_cleaned_df[col] = bank_cleaned_df[col].replace('unknown', value_mode)

### STUDENT CODE until HERE

print("\nVerifying 'unknown' removal:")
verification_passed = True  #es ist immer true?????
for col in cols_with_unknown:
    unknown_count = (bank_cleaned_df[col] == "unknown").sum()
    print(f"  - Column '{col}' 'unknown' count after imputation: {unknown_count}")
    try:
        assert (
            unknown_count == 0
        ), f"Assertion Failed: Column '{col}' still contains 'unknown' values!"
    except AssertionError as e:
        print(e)
        verification_passed = False

if verification_passed:
    print("\n'Unknown' values handled successfully.")
else:
    print("\nVerification failed: Some 'unknown' values remain.")

--- Handling 'Unknown' Values (Imputing with Mode) ---
['job', 'education', 'contact', 'poutcome']

Imputing 'unknown' with mode:

Verifying 'unknown' removal:
  - Column 'job' 'unknown' count after imputation: 0
  - Column 'education' 'unknown' count after imputation: 0
  - Column 'contact' 'unknown' count after imputation: 0
  - Column 'poutcome' 'unknown' count after imputation: 0

'Unknown' values handled successfully.


## 2.2 Handling Problematic Features

As discussed during EDA (Unit 1) and reiterated here, some features require special attention:

*   **'duration':** This feature represents the duration of the *last* contact with the client *during the current marketing campaign*. While highly predictive of whether the client subscribed *in that instance*, its value is only known *after* the call is completed. Including it in a model intended to predict *future* subscriptions or to decide *who* to call would constitute **data leakage**. The model would learn a pattern that relies on information unavailable at the time of prediction in a real-world scenario. Therefore, for building a realistic predictive model, **we must remove this feature.**

*   **'pdays':** This feature indicates the number of days that passed by after the client was last contacted from a previous campaign (-1 means the client was not previously contacted). While the -1 value has a special meaning, it's numerical. For this unit, we'll keep it as is. A potential feature engineering step could be to create a binary feature `was_contacted_previously` (1 if `pdays != -1`, 0 otherwise) and perhaps treat the actual day count separately, but we'll maintain simplicity here.

<div class="alert alert-block alert-success">
<b>Task 2.2: Remove 'duration' Column (0.5 pt)</b>

<ul>
    <li> Remove the 'duration' column from the `bank_cleaned_df` DataFrame.</li>
    <li> Verify its removal by printing the list of remaining columns.</li>
</ul>
</div>

In [144]:
print("--- Handling Problematic Features ---")

if "duration" in bank_cleaned_df.columns:
    ### STUDENT CODE HERE (0.5 pt)
    bank_cleaned_df.drop(columns="duration", inplace=True)

    ### STUDENT CODE until HERE

    assert "duration" not in bank_cleaned_df.columns
else:
    print("  - 'duration' column not found or already removed.")

print("\nRemaining columns after handling problematic features:")
print(bank_cleaned_df.columns.tolist())

--- Handling Problematic Features ---

Remaining columns after handling problematic features:
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'campaign', 'pdays', 'previous', 'poutcome', 'y']


## 2.3 Outlier Discussion

Numerical features like 'balance' (client account balance) or 'campaign' (number of contacts during this campaign) might contain values that are significantly different from the majority of the observations. These are known as **outliers**.

**How to Detect?**
*   **Visual Inspection:** Histograms and box plots (from Unit 1 or Unit 3) can reveal potential outliers (e.g., points far outside the whiskers of a box plot).
*   **Statistical Methods:**
    *   **Z-Score:** Calculate how many standard deviations a data point is from the mean. Points beyond a threshold (e.g., |Z| > 3) are often considered outliers. Assumes data is roughly normally distributed.
    *   **Interquartile Range (IQR) Method:** Define outliers as points falling below `Q1 - 1.5 * IQR` or above `Q3 + 1.5 * IQR`, where Q1 and Q3 are the 25th and 75th percentiles, and IQR = Q3 - Q1. More robust to skewed distributions than the Z-score method.

**How to Handle?**
*   **Keep:** If outliers represent genuine, albeit rare, occurrences and your chosen model is robust to them (e.g., tree-based models), you might keep them.
*   **Cap (Winsorize):** Replace outliers with the nearest "acceptable" value (e.g., the boundary calculated by the IQR method). Reduces extremity while retaining the sample.
*   **Transform:** Apply mathematical functions (e.g., log, square root, Box-Cox) to reduce the impact of extreme values by compressing the range. Often helps with right-skewed data.
*   **Remove:** Delete rows containing outliers. Use with extreme caution, as it means discarding data and potentially biasing results if the outliers weren't simply errors.
*   **Use Robust Models:** Employ algorithms inherently less sensitive to outliers (e.g., Random Forests, Gradient Boosting Trees, models using robust loss functions like Huber loss).

**Decision for this Unit:** As mentioned before, handling outliers effectively requires careful analysis. For simplicity in this foundational lab, we will **acknowledge** their potential presence (e.g., the wide range and high standard deviation of 'balance') but **will not apply specific outlier treatment techniques** to `bank_cleaned_df`. We will proceed with the data as is after handling 'unknown' and removing 'duration'.

<div class="alert alert-block alert-info">
    <b>Further Reading on Outliers:</b>
    <ul>
        <li><a href="https://www.scribbr.com/statistics/outliers/" target="_blank">Scribbr: Outliers | What They Are & How To Deal With Them</a></li>
        <li><a href="https://medium.com/data-science/ways-to-detect-and-remove-the-outliers-404d16608dba" target="_blank">Ways to Detect and Remove the Outliers (Medium)</a></li>
    </ul>
</div>

## 2.4 Categorical Feature Encoding

Most machine learning algorithms operate on numerical data. Our dataset contains numerous categorical features (represented as 'object' dtype) which need to be converted into a numerical format. These include features like 'job', 'marital', 'education', 'default' (credit in default?), 'housing' (housing loan?), 'loan' (personal loan?), 'contact' (communication type), 'month', and 'poutcome' (previous outcome).

There are several ways to encode categorical features. The appropriate method often depends on whether the categories have an inherent order (ordinal) or not (nominal).

<div class="alert alert-block alert-info">
    <b>Encoding Ordinal vs. Nominal Features:</b>
    <ul>
        <li><b>Ordinal Encoding:</b> Assigns a unique integer to each category (e.g., 'small' -> 0, 'medium' -> 1, 'large' -> 2, or 'negative' -> -1, 'neutral' -> 0, 'positive' -> 1). This method is suitable <b>only</b> when the categories have a meaningful, inherent order that you want the model to potentially learn from (like education levels or size categories).</li>
        <li><b>One-Hot Encoding (OHE):</b> Creates a new binary (0/1) column for each category. This is generally preferred for <b>nominal</b> features (like 'job', 'marital status', 'colour') where there is no intrinsic order. Using simple integers (like 0, 1, 2) for nominal features can mislead the model into thinking there's a mathematical relationship or ranking between categories that doesn't actually exist (e.g., assuming 'management'=2 is somehow "twice" 'technician'=1).</li>
    </ul>
    Since most of our categorical features ('job', 'marital', 'contact', etc.) are nominal, we will use One-Hot Encoding to avoid imposing an artificial order.
</div>

We will use **One-Hot Encoding (OHE)**. This technique transforms each categorical feature into multiple new binary (0 or 1) features, one for each unique category within the original feature.

**Example:**
If a 'marital' column has categories `['married', 'single', 'divorced']`, OHE creates three new columns:
*   `marital_married`
*   `marital_single`
*   `marital_divorced`

A sample where the original value was 'single' would have: `marital_married=0`, `marital_single=1`, `marital_divorced=0`.

**Handling Multicollinearity (`drop_first=True`):**
Notice that if we know the values for `marital_married` and `marital_single`, we automatically know the value for `marital_divorced` (if it's not 1 in the first two, it must be 1 in the third, assuming these are the only categories). This perfect linear relationship between the dummy columns is called **multicollinearity**. While many algorithms can handle it, it can cause issues for some (especially linear models interpreting coefficients). A common practice to avoid this is to drop one of the dummy columns for each original feature (e.g., drop `marital_divorced`). The information is still implicitly present. We achieve this using the `drop_first=True` parameter in `pandas.get_dummies`.

<div class="alert alert-block alert-success">
<b>Task 2.3: One-Hot Encode Categorical Features (3 pts)</b>

<ul>
    <li> Identify all remaining columns in <code>bank_cleaned_df</code> with an 'object' data type (these are our categorical features). Remember to exclude the target variable 'y' if it's still in the DataFrame.</li>
    <li> Use <code>pd.get_dummies()</code> to perform One-Hot Encoding on these identified categorical columns within the <code>bank_cleaned_df</code>.
    </li>
    <li> Print the shape of the new <code>bank_encoded_df</code> to see how dimensionality has increased.</li>
    <li> Display the first few rows of <code>bank_encoded_df</code> to observe the newly created binary columns.</li>
    <li> Print the list of all columns in <code>bank_encoded_df</code>.</li>
</ul>
</div>

In [145]:
print("--- Categorical Feature Encoding (One-Hot Encoding) ---")

# Identify categorical columns for One-Hot Encoding
categorical_cols: List[str] = []
### STUDENT CODE HERE (0.5 pt)
categorical_cols = bank_cleaned_df.select_dtypes(include="object").columns
if 'y' in categorical_cols:
    categorical_cols = categorical_cols.drop("y")

#categorical_cols = categorical_cols.tolist()
### STUDENT CODE until HERE
print(f"Categorical columns to be one-hot encoded: {categorical_cols}")

# Apply One-Hot Encoding using pd.get_dummies
bank_encoded_df = bank_cleaned_df.copy()  # Work on a copy

### STUDENT CODE HERE (2 pts)
bank_encoded_df = pd.get_dummies(data=bank_cleaned_df, columns=categorical_cols)

### STUDENT CODE until HERE

#Below, proceed to print out the resulting shape, first 5 columns, and new columns.
print("\nDataFrame after One-Hot Encoding:")
### STUDENT CODE HERE (0.5 pts)
display(bank_encoded_df.head())
print(bank_encoded_df.columns.tolist())
### STUDENT CODE until HERE

--- Categorical Feature Encoding (One-Hot Encoding) ---
Categorical columns to be one-hot encoded: Index(['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact',
       'month', 'poutcome'],
      dtype='object')

DataFrame after One-Hot Encoding:


,age,balance,day,campaign,pdays,previous,y,job_969,job_admin.,job_blue-collar,...,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_3705,poutcome_failure,poutcome_other,poutcome_success
0,30,1787,19,1,-1,0,no,False,False,False,...,False,False,False,False,True,False,True,False,False,False
1,33,4789,11,1,339,4,no,False,False,False,...,False,False,True,False,False,False,False,True,False,False
2,35,1350,16,1,330,1,no,False,False,False,...,False,False,False,False,False,False,False,True,False,False
3,30,1476,3,4,-1,0,no,False,False,False,...,True,False,False,False,False,False,True,False,False,False
4,59,0,5,1,-1,0,no,False,False,True,...,False,False,True,False,False,False,True,False,False,False


['age', 'balance', 'day', 'campaign', 'pdays', 'previous', 'y', 'job_969', 'job_admin.', 'job_blue-collar', 'job_entrepreneur', 'job_housemaid', 'job_management', 'job_retired', 'job_self-employed', 'job_services', 'job_student', 'job_technician', 'job_unemployed', 'marital_divorced', 'marital_married', 'marital_single', 'education_2306', 'education_primary', 'education_secondary', 'education_tertiary', 'default_no', 'default_yes', 'housing_no', 'housing_yes', 'loan_no', 'loan_yes', 'contact_2896', 'contact_cellular', 'contact_telephone', 'month_apr', 'month_aug', 'month_dec', 'month_feb', 'month_jan', 'month_jul', 'month_jun', 'month_mar', 'month_may', 'month_nov', 'month_oct', 'month_sep', 'poutcome_3705', 'poutcome_failure', 'poutcome_other', 'poutcome_success']


## 2.5 Numerical Feature Scaling

Many machine learning algorithms are sensitive to the scale of input features. Features with larger values or wider ranges might disproportionately influence the model compared to features with smaller values. Examples include:
*   **Distance-based algorithms (e.g., KNN, K-Means, SVM with certain kernels):** Distances are directly affected by feature scales.
*   **Gradient-based algorithms (e.g., Linear/Logistic Regression, Neural Networks):** Scaling helps gradient descent converge faster and more stably by ensuring steps taken for different feature weights are more balanced.

We will use **Standardization (Z-score Scaling)**. This common technique rescales features so they have the properties of a standard normal distribution: a **mean of 0** and a **standard deviation of 1**.

The formula for standardization is:
`X_scaled = (X - X_mean) / X_stddev`

Where `X_mean` is the mean of the feature and `X_stddev` is its standard deviation.

**Important:** We must apply scaling *only* to the original numerical columns. The binary (0/1) columns created by One-Hot Encoding do not need scaling (they are already on a comparable scale).

<div class="alert alert-block alert-success">
<b>Task 2.4: Standardize Numerical Features (2 pts)</b>

<ul>
    <li> First, identify the names of the original numerical columns that are present in <i>bank_encoded_df</i>. (These are the columns that were numerical <b>before</b> OHE).</li>
    <li> Instantiate <code>sklearn.preprocessing.StandardScaler</code>.</li>
    <li> <b>Fit</b> the scaler <b>only</b> on the numerical columns identified in the first step, using the data from <i>bank_encoded_df</i>. The `fit` method learns the mean and standard deviation for each of these columns.</li>
    <li> <b>Transform</b> these same numerical columns using the fitted scaler. This applies the standardization formula.</li>
    <li> Replace the original numerical columns in <i>bank_encoded_df</i> with their scaled versions. Be careful to maintain the DataFrame structure and indices.</li>
    <li> Verify the scaling by printing the `mean` and `std` of the scaled columns (they should be close to 0 and 1, respectively).</li>
</ul>
</div>

In [146]:
print("--- Numerical Feature Scaling (Standardization) ---")

# Identify the numerical columns to be scaled
numerical_cols_to_scale: List[str] = []
### STUDENT CODE HERE (0.5 pt)
numerical_cols_to_scale= bank_encoded_df.select_dtypes(include="number").columns
numerical_cols_to_scale = numerical_cols_to_scale.tolist()
### STUDENT CODE until HERE
print(f"Numerical columns to be scaled: {numerical_cols_to_scale}")

if numerical_cols_to_scale:
    scaler: StandardScaler = None

    ### STUDENT CODE HERE (2 pts) 
    scaler = StandardScaler()

    scaler.fit(bank_encoded_df[numerical_cols_to_scale])

    scaled_values  = scaler.transform(bank_encoded_df[numerical_cols_to_scale])
    
    bank_encoded_df[numerical_cols_to_scale] = scaled_values
    ### STUDENT CODE until HERE 

    print("Verifying scaling by checking mean and std dev of scaled columns:")
    display(
        bank_encoded_df[numerical_cols_to_scale]
        .describe()
        .loc[["mean", "std"]]
        .round(2)
    )
else:
    print("No numerical columns identified for scaling.")

--- Numerical Feature Scaling (Standardization) ---
Numerical columns to be scaled: ['age', 'balance', 'day', 'campaign', 'pdays', 'previous']
Verifying scaling by checking mean and std dev of scaled columns:


,age,balance,day,campaign,pdays,previous
mean,-0.0,-0.0,0.0,-0.0,-0.0,0.0
std,1.0,1.0,1.0,1.0,1.0,1.0


## 2.6 Preparing Final Data for Modeling

We have now completed the main cleaning and preprocessing steps for this lab:
*   Handled 'unknown' values.
*   Removed the 'duration' feature.
*   One-Hot Encoded categorical features.
*   Standardized numerical features.

The `bank_encoded_df` DataFrame now contains our processed feature set. The final step in this unit is to:
1.  Separate the processed features (X) from the target variable (y).
2.  Ensure the target variable 'y' is numerically encoded (1 for 'yes', 0 for 'no').
3.  Convert the features and target into NumPy arrays, which is the standard input format for most Scikit-learn models.
4.  Save these final arrays (`X_processed`, `y_processed`) to disk so they can be easily loaded in subsequent units without rerunning all preprocessing steps.

<div class="alert alert-block alert-success">
<b>Task 2.5: Create and Save Final Processed Arrays (1 pt)</b>

<ol>
    <li> Create the feature matrix <i>X_processed</i> by selecting all columns from <i>bank_encoded_df</i> <b>except</b> the original target column <i>y</i>.</li>
    <li> Create the target vector <i>y_processed<i> by selecting the original <i>y</i> column from the original <i>bank_df</i> and mapping <i>yes</i> to 1 and <i>no</i> to 0.</li>
    <li> Convert both <i>X_processed</i> and <i>y_processed</i> into NumPy arrays using the <code>.values</code> attribute.</li>
    <li> Define file paths (`data/bank_X_processed.npy`, `data/bank_y_processed.npy`).</li>
    <li> Save the <i>X_processed</i> and <i>y_processed</i> NumPy arrays using <code>np.save()</code>.</li>
</ol>
</div>

In [147]:
print("--- Preparing and Saving Final Processed Data ---")

# Initialize final variables
X_processed: np.ndarray = None
y_processed: np.ndarray = None
X_processed_df: pd.DataFrame = None
y_processed_series: pd.Series = None

if "bank_encoded_df" in locals() and not bank_encoded_df.empty and bank_df is not None:

    ### STUDENT CODE HERE (1 pt) 
    X_processed = bank_encoded_df.drop(columns='y')
    y_processed = bank_cleaned_df["y"].replace(to_replace=['yes','no'],value=[1,0])

    X_processed = X_processed.values
    y_processed = y_processed.values

    X_file_path = 'data/bank_X_processed.npy'
    y_file_path = 'data/bank_y_processed.npy'

    np.save(X_file_path, X_processed)
    np.save(y_file_path, y_processed)
    ### STUDENT CODE until HERE 
else:
    print(
        "Input DataFrames ('bank_encoded_df', 'bank_df') not available, cannot prepare or save final data."
    )

--- Preparing and Saving Final Processed Data ---


C:\Users\arnes\AppData\Local\Temp\ipykernel_13064\3783536761.py:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_processed = bank_cleaned_df["y"].replace(to_replace=['yes','no'],value=[1,0])


## Unit 2 Summary & Deep Dive Topics

In this unit, we performed crucial data cleaning and preprocessing steps on the Bank Marketing dataset. We handled 'unknown' values by imputing the mode, removed the problematic 'duration' feature to prevent data leakage, applied One-Hot Encoding to convert categorical features into a numerical format, and used Standardization to scale numerical features. Finally, we separated our processed features (`X_processed`) and the numerically encoded target variable (`y_processed`), saving them as NumPy arrays, making the data ready for the next stages of the machine learning workflow.

### Deep Dive Topics

#### 1. Advanced Imputation Techniques:


While we used simple mode imputation for <i>unknown</i> values, more sophisticated methods exist for handling missing numerical or categorical data when <i>unknown</i> isn't the only indicator or when data is missing completely (NaN). These often leverage relationships *between* features:

*   **K-Nearest Neighbors (KNN) Imputation:** This method finds the <i>k</i> nearest neighboring samples (based on the features *without* missing values) to a sample with a missing value. It then imputes the missing value based on the values of that feature in its neighbors (e.g., using the mean/median for numerical data or the mode for categorical data). This can capture more complex relationships than simple mean/median/mode imputation.
    *   *Check out:*
        *   Scikit-learn Documentation: [`sklearn.impute.KNNImputer`](https://scikit-learn.org/stable/modules/generated/sklearn.impute.KNNImputer.html)
        *   Tutorial/Article: [KNN Imputation: The Complete Guide - Medium](https://medium.com/@tarangds/knn-imputation-the-complete-guide-146f932870a7)
*   **Iterative Imputation (Multivariate Imputation by Chained Equations - MICE):** This is a more advanced technique where each feature with missing values is treated as a target variable (y), and a regression model (like linear regression, RandomForest, etc.) is trained using the other features (X) to predict the missing values. This process is done iteratively, refining the imputations in each round until they converge. It can capture complex interactions and non-linear relationships.
    *   *Check out:*
        *   Scikit-learn Documentation: [`sklearn.impute.IterativeImputer`](https://scikit-learn.org/stable/modules/generated/sklearn.impute.IterativeImputer.html)
        *   Overview Article: [Iterative Imputation - Towards Data Science](https://towardsdatascience.com/iterative-imputation-with-scikit-learn-8f3eb22b1a38)

**Choosing an Imputation Method:** The best method often depends on the dataset size, the nature of the missing data (MCAR, MAR, MNAR), the relationships between features, and computational resources. Simple methods are fast baselines, while advanced methods can potentially yield better results if missingness is related to other features, but they are computationally more expensive.

#### 2. Feature Engineering vs. Self-Supervised Learning (SSL):

*   **Manual Feature Engineering:** This is what we partially did by deciding how to handle 'unknown' or considering binning 'age'. It involves using domain knowledge or insights from EDA to create new features from existing ones (e.g., creating interaction terms like `age * balance`, polynomial features, ratios like `balance / age`, or grouping categories). The goal is to manually craft features that make the underlying patterns more accessible to the learning algorithm. This requires significant effort, intuition, and can be domain-specific.

*   **Self-Supervised Learning (SSL):** This is a paradigm within machine learning that aims to learn meaningful **feature representations** directly from data *without* relying on explicit human-provided labels (like 'yes'/'no' for subscription). Instead, it creates "pseudo-labels" or supervisory signals *from the input data itself*.
    *   **How it works (Conceptual):** SSL tasks often involve predicting a hidden or corrupted part of the input from the rest. Examples include predicting masked words in text (like BERT) or learning representations by contrasting augmented views of images (like SimCLR or MoCo).
    *   **Goal:** The objective is to force the model (often a deep neural network) to learn high-level, semantic features about the data structure, distribution, and relationships during a "pre-training" phase on vast amounts of *unlabeled* data. These learned representations (features) can then be used for downstream tasks (like classification) with much less labeled data required for fine-tuning, potentially leading to better performance and generalization.
    *   **Relation to Feature Engineering:** SSL can be seen as **automating the feature representation learning process**. Instead of humans manually designing features, the model learns them by solving auxiliary tasks on unlabeled data. This is particularly powerful because unlabeled data is often far more abundant than labeled data.
    *   *Check out:*
        *   General Overview: [What is Self-Supervised Learning? - IBM](https://www.ibm.com/think/topics/self-supervised-learning)
        *   NLP Example (BERT): [BERT Explained: State of the art language model for NLP - Towards Data Science](https://towardsdatascience.com/bert-explained-state-of-the-art-language-model-for-nlp-f8b21a9b6270) & [Hugging Face Transformers Library](https://huggingface.co/docs/transformers/index)
        *   Vision Example (SimCLR): [SimCLR Framework - Google AI Blog Post](https://ai.googleblog.com/2020/04/advancing-self-supervised-and-semi.html) & [VISSL Library (Facebook AI)](https://github.com/facebookresearch/vissl)
        *   Frameworks: [PyTorch Lightning Bolts (includes SSL models)](https://lightning.ai/docs/pytorch/2.1.1/ecosystem/bolts.html)
        *   Comparison AI Stackexchange answer: [Where do the feature extraction and representation learning differ?](https://ai.stackexchange.com/questions/27996/where-do-the-feature-extraction-and-representation-learning-differ)

While manual feature engineering is still valuable, especially for tabular data, SSL represents a significant shift towards letting models discover powerful features themselves, particularly when large unlabeled datasets are available.